# Où partir selon la météo ? Webscrapping and AWS
## Introduction

**Pourquoi ce projet ?** \
Lorsqu'on prévoit un séjour en France, on consulte souvent la météo et Booking.com séparément — mais on ne les relie jamais réellement. Ce projet a pour objectif de **croiser ces deux sources d'information** pour recommander les meilleures villes où partir, **en fonction des conditions météo prévues et de la qualité des hôtels disponibles**.

En combinant **prévisions météo à 7 jours** et **notes des hôtels**, nous construisons un **indice de confort** permettant d’orienter rapidement les utilisateurs vers les **meilleures destinations possibles** du moment.  
L'idée est de rendre **l'exploration de Booking plus intelligente**, plus data-driven.

**À qui cela s’adresse ?** 
- Aux voyageurs qui veulent **partir sans galérer à croiser les données**
- À une plateforme comme Kayak qui veut **booster la conversion sur des offres météo-friendly**

**Goal:** \
Ce projet vise à aider les utilisateurs de Kayak à choisir la meilleure destination de vacances en France en se basant sur la météo prévue pour les 7 prochains jours, tout en leur proposant les meilleurs hébergements disponibles sur Booking.com.

**Étapes:**
1. Collecter les données météo à 7 jours pour les 35 principales villes françaises
2. Évaluer chaque ville selon un indice météo personnalisé
3. Identifier les 5 villes françaises offrant les meilleures conditions météo dans les 7 prochains jours
4. Scrapper les Hotels dans les 35 villes de départ
5. Exporte les données reccueillies vers un bucket aws
6. Créer une base de donnée sur AWS RDS
6. Utiliser les donnée dans la database sur AWS RDS pour créer une carte interactive Mapbox pour visualiser les 5 villes avec le meilleur indice météo et les 20 hotels les mieux noté dans ces 5 villes 

In [47]:
import pandas as pd
import requests
import json
import os
import boto3
import re
from cities import cities
from dotenv import load_dotenv
from sqlalchemy import create_engine
import numpy as np
import plotly.express as px

Création d'un environnement sécurisé pour mes accès aws

In [ ]:
load_dotenv(dotenv_path=os.path.expanduser("~/.secret_access/kayak_aws"))


True

## 1. Weather data
**Goal:** \
Savoir quelles villes dasn les 35 villes de départ on une meilleure météo dans les 7 prochains jours.

**Actions:**
* Récuperer les données de localisation de 35 villes
* Récupérer les données de météo pour ces villes
* Création d'un score de météo, dans le but de pouvoir classer les villes par score
* Enregistrement des données dans un csv

### Pourquoi un score météo ?
L’objectif est de proposer à l’utilisateur final une recommandation simple et lisible. 
Le score météo permet de comparer les conditions météorologiques des villes de manière agrégée, 
en tenant compte de la température, du vent et des précipitations.

In [3]:
# Get the latitude and longitude of the cities
cities_location = []

headers = {
    "User-Agent": "QHA"
}

for city in cities:
    r_location = requests.get(f"https://nominatim.openstreetmap.org/search?format=json&city={city}", headers=headers)
    location_data = r_location.json()

    current_city = {
        "name": location_data[0]['name'],
        "lat": location_data[0]['lat'],
        "lon": location_data[0]['lon'],
    }

    cities_location.append(current_city)
print(cities_location)



[{'name': 'Mont-Saint-Michel', 'lat': '46.7798558', 'lon': '-75.3362610'}, {'name': 'St. Malo', 'lat': '49.3146950', 'lon': '-96.9538228'}, {'name': 'Bayeux', 'lat': '49.2764624', 'lon': '-0.7024738'}, {'name': 'Le Havre', 'lat': '49.4938975', 'lon': '0.1079732'}, {'name': 'Rouen', 'lat': '49.4404591', 'lon': '1.0939658'}, {'name': 'Paris', 'lat': '48.8588897', 'lon': '2.3200410'}, {'name': 'Amiens', 'lat': '49.8941708', 'lon': '2.2956951'}, {'name': 'Lille', 'lat': '50.6365654', 'lon': '3.0635282'}, {'name': 'Strasbourg', 'lat': '48.5846140', 'lon': '7.7507127'}, {'name': 'Château du Haut-Kœnigsbourg', 'lat': '48.2495226', 'lon': '7.3454923'}, {'name': 'Colmar', 'lat': '48.0777517', 'lon': '7.3579641'}, {'name': 'Eguisheim', 'lat': '48.0447968', 'lon': '7.3079618'}, {'name': 'Besançon', 'lat': '47.2380222', 'lon': '6.0243622'}, {'name': 'Dijon', 'lat': '47.3215806', 'lon': '5.0414701'}, {'name': 'Annecy', 'lat': '45.8992348', 'lon': '6.1288847'}, {'name': 'Grenoble', 'lat': '45.187560

In [4]:
df_location = pd.DataFrame(cities_location)
df_location

,name,lat,lon
0,Mont-Saint-Michel,46.7798558,-75.3362610
1,St. Malo,49.3146950,-96.9538228
2,Bayeux,49.2764624,-0.7024738
3,Le Havre,49.4938975,0.1079732
4,Rouen,49.4404591,1.0939658
5,Paris,48.8588897,2.3200410
6,Amiens,49.8941708,2.2956951
7,Lille,50.6365654,3.0635282
8,Strasbourg,48.5846140,7.7507127
9,Château du Haut-Kœnigsbourg,48.2495226,7.3454923


In [5]:
api_weather_key = os.getenv("open_weather_api_key")

params = {
    "appid": api_weather_key,
    "units": "metric",
    "lang": "fr",
    "cnt": 5,
}

lat = 44.8333
lon = -0.5667
cnt = 16

url_test = f"https://api.openweathermap.org/data/2.5/forecast?lat=44.8333&lon=-0.5667&units=metric&lang=fr&cnt=7&appid={api_weather_key}"

r = requests.get(url_test)
data = r.json()

data['list']



[{'dt': 1747818000,
  'main': {'temp': 16.31,
   'feels_like': 16.11,
   'temp_min': 16.31,
   'temp_max': 18.55,
   'pressure': 1020,
   'sea_level': 1020,
   'grnd_level': 1014,
   'humidity': 81,
   'temp_kf': -2.24},
  'weather': [{'id': 802,
    'main': 'Clouds',
    'description': 'partiellement nuageux',
    'icon': '03d'}],
  'clouds': {'all': 46},
  'wind': {'speed': 4.29, 'deg': 266, 'gust': 5.79},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'd'},
  'dt_txt': '2025-05-21 09:00:00'},
 {'dt': 1747828800,
  'main': {'temp': 18.42,
   'feels_like': 18.12,
   'temp_min': 18.42,
   'temp_max': 20.04,
   'pressure': 1019,
   'sea_level': 1019,
   'grnd_level': 1013,
   'humidity': 69,
   'temp_kf': -1.62},
  'weather': [{'id': 803,
    'main': 'Clouds',
    'description': 'nuageux',
    'icon': '04d'}],
  'clouds': {'all': 73},
  'wind': {'speed': 5.64, 'deg': 265, 'gust': 6.84},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'd'},
  'dt_txt': '2025-05-21 12:00:00'},
 {

In [6]:
all_weather = []
weather_params = {
    "appid": api_weather_key,
    "units": "metric",
    "lang": "fr",
    "cnt": 7,
}

for index,weather in enumerate(cities_location):
    temp_score = 0
    rain = 0
    wind = 0
    wind_score = 0
    rain_score = 0
    
    r_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={weather['lat']}&lon={weather['lon']}", params=weather_params)
    current_weather = r_weather.json()['list']

    for i in range(len(current_weather)):
        temp_score = round(temp_score + (current_weather[i]['main']['feels_like'] / len(current_weather)), 2)
        rain = current_weather[i].get('rain', 0)
        
        if rain != 0:
            rain_key = list(current_weather[i]['rain'].keys())[0]
            match = re.search(r'\d+', rain_key)
            hours = 0

            if match:
                hours = int(match.group())

            rain_daily_score = hours * current_weather[i]['rain'][rain_key]
            rain_score = round(rain_score + (rain_daily_score / len(current_weather)), 2)
        else:
            rain_score = rain_score + 0

        wind = round(wind + (current_weather[i]['wind']['speed'] / len(current_weather)), 2)

        beaufort_score = 0 # Beaufort scale
        if current_weather[i]['wind']['speed'] == 0:
            beaufort_score = 0
        elif current_weather[i]['wind']['speed'] < 5:
            beaufort_score = 1
        elif current_weather[i]['wind']['speed'] < 11:
            beaufort_score = 2
        elif current_weather[i]['wind']['speed'] < 19:
            beaufort_score = 3
        elif current_weather[i]['wind']['speed'] < 28:
            beaufort_score = 4
        elif current_weather[i]['wind']['speed'] < 38:
            beaufort_score = 5
        elif current_weather[i]['wind']['speed'] < 49:
            beaufort_score = 6
        elif current_weather[i]['wind']['speed'] < 61:
            beaufort_score = 7
        elif current_weather[i]['wind']['speed'] < 74:
            beaufort_score = 8
        elif current_weather[i]['wind']['speed'] < 88:
            beaufort_score = 9
        elif current_weather[i]['wind']['speed'] < 102:
            beaufort_score = 10
        elif current_weather[i]['wind']['speed'] < 117:
            beaufort_score = 11
        elif current_weather[i]['wind']['speed'] > 117:
            beaufort_score = 12

        if current_weather[i]['main']['feels_like'] < 25 or current_weather[i]['wind']['speed'] > 30:
            wind_penalty = beaufort_score
        else:
            wind_penalty = 0

        wind_score = round(wind_score + (wind_penalty / len(current_weather)), 2)

    weather_score = round(temp_score - rain_score - wind_score, 3)


    current_weather_data = {
        "index" : index,
        "name" : weather['name'],
        "temperature_mean" : temp_score,
        "rain_mean" : rain_score,
        "wind_score" : wind_score,
        "score" : weather_score,
    }
    all_weather.append(current_weather_data)
       
all_weather


[{'index': 0,
  'name': 'Mont-Saint-Michel',
  'temperature_mean': 7.56,
  'rain_mean': 0.83,
  'wind_score': 0.98,
  'score': 5.75},
 {'index': 1,
  'name': 'St. Malo',
  'temperature_mean': 10.46,
  'rain_mean': 0.58,
  'wind_score': 1.43,
  'score': 8.45},
 {'index': 2,
  'name': 'Bayeux',
  'temperature_mean': 10.83,
  'rain_mean': 5.82,
  'wind_score': 0.98,
  'score': 4.03},
 {'index': 3,
  'name': 'Le Havre',
  'temperature_mean': 11.3,
  'rain_mean': 4.73,
  'wind_score': 0.98,
  'score': 5.59},
 {'index': 4,
  'name': 'Rouen',
  'temperature_mean': 12.08,
  'rain_mean': 4.66,
  'wind_score': 0.98,
  'score': 6.44},
 {'index': 5,
  'name': 'Paris',
  'temperature_mean': 11.5,
  'rain_mean': 4.64,
  'wind_score': 0.98,
  'score': 5.88},
 {'index': 6,
  'name': 'Amiens',
  'temperature_mean': 10.46,
  'rain_mean': 0.55,
  'wind_score': 0.98,
  'score': 8.93},
 {'index': 7,
  'name': 'Lille',
  'temperature_mean': 11.56,
  'rain_mean': 0.82,
  'wind_score': 1.13,
  'score': 9.61},

In [7]:
df_weather = pd.DataFrame(all_weather)
df_weather = df_weather.sort_values(by='score', ascending=False)
df_weather

,index,name,temperature_mean,rain_mean,wind_score,score
22,22,Avignon,20.17,0.16,1.29,18.72
24,24,Nîmes,19.84,0.00,1.14,18.70
23,23,Uzès,19.24,0.19,1.28,17.77
20,20,Marseille,20.14,0.39,2.03,17.72
27,27,Collioure,19.05,0.37,0.98,17.70
21,21,Aix-en-Provence,19.14,0.14,1.43,17.57
25,25,Aigues-Mortes,19.73,0.31,2.03,17.39
18,18,Bormes-les-Mimosas,18.80,0.00,1.88,16.92
19,19,Cassis,19.24,0.49,2.03,16.72
26,26,Saintes-Maries-de-la-Mer,18.66,0.35,2.03,16.28


In [8]:
df_weather.to_csv("export/weather.csv", index=False)

## 2. Hotel data
**Goal:** \
Scrapper les données des hôtels dans les villes via le site booking.com. 

**Actions:**
* Création d'un boucle de récupération d'url
* Récupérer les données de hotel sur booking.com via scrappy
* Enregistrement des données dans un CSV

In [9]:
!python hotel.py

2025-05-21 09:25:34 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: scrapybot)
2025-05-21 09:25:34 [scrapy.utils.log] INFO: Versions: lxml 5.2.1.0, libxml2 2.13.1, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 23.10.0, Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:28:27) [Clang 14.0.6 ], pyOpenSSL 24.2.1 (OpenSSL 3.0.15 3 Sep 2024), cryptography 43.0.0, Platform macOS-10.16-x86_64-i386-64bit
2025-05-21 09:25:34 [scrapy.addons] INFO: Enabled addons:
[]
2025-05-21 09:25:34 [py.warnings] WARNING: /opt/anaconda3/lib/python3.12/site-packages/scrapy/utils/request.py:254: ScrapyDeprecationWarning: '2.6' is a deprecated value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting.

It is also the default value. In other words, it is normal to get this warning if you have not defined a value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting. This is so for backward compatibility reasons, but it will change in a future version of Scrapy.

See the documentati

## 3. Merger les données en un seul dataframe
**Goal:** \
Stocker toutes les données dans un seul dataframe pour pouvoir les utiliser plus tard.

**Actions:**
* Importation des données
* Fusionner les données dans un seul dataframe
* Enregistrement des données dans un CSV

In [10]:
with open("export/hotels.json", "r") as f:
    hotel_data = json.load(f)

df_hotel = pd.DataFrame(hotel_data)
df_hotel = df_hotel.rename(columns={""
    "name": "hotel_name",
    "link": "hotel_link",
    "latitude": "lat",
    "longitude": "lon",
    "description": "desc",
})
df_hotel[df_hotel["city_id"]== 0]

,city_id,city,hotel_name,hotel_link,lat,lon,desc,stars,rating
363,0,Mont Saint Michel,"Le Petit Moulin Rouge, B&B, SDB en-suite, park...",https://www.booking.com/hotel/fr/chambre-cosy-...,48.5963383,-1.5055117,"Situé à Beauvoir, l’établissement Le Petit Mou...",0,"9,5"
364,0,Mont Saint Michel,Maison au pied du Mont Saint Michel 2,https://www.booking.com/hotel/fr/maison-au-pie...,48.6156117,-1.4885121,"Offrant une vue sur le jardin, l’hébergement M...",0,"9,3"
365,0,Mont Saint Michel,Roulotte au pied du Mont Saint Michel,https://www.booking.com/hotel/fr/roulotte-au-p...,48.613969646033,-1.486292458336,L’hébergement Roulotte au pied du Mont Saint M...,0,"9,5"
366,0,Mont Saint Michel,Auberge de la Baie,https://www.booking.com/hotel/fr/auberge-de-la...,48.61599317655586,-1.4882531762123108,"Situé dans la campagne normande, cet hôtel dis...",2,"8,3"
367,0,Mont Saint Michel,Les Pres-Salés,https://www.booking.com/hotel/fr/les-pra-s-sal...,48.62422199651231,-1.4457803964614868,L’établissement Les Pres-Salés vous accueille ...,0,"8,9"
368,0,Mont Saint Michel,Mon Saint Michel,https://www.booking.com/hotel/fr/mon-saint-mic...,48.613627,-1.485791,"Offrant une vue sur le jardin, l’établissement...",0,"8,8"
369,0,Mont Saint Michel,A l ombre du Mont Saint Michel,https://www.booking.com/hotel/fr/a-l-ombre-du-...,48.6156258,-1.4651207,"Situé à Huisnes-sur-Mer, l’établissement A l o...",0,"9,5"
370,0,Mont Saint Michel,La Jacotière,https://www.booking.com/hotel/fr/la-jacotia-re...,48.61411436123917,-1.5043142437934875,"Implantée à Ardevon, la maison d'hôtes La Jaco...",0,"9,2"
371,0,Mont Saint Michel,Gîte La Mouette de 4 personnes,https://www.booking.com/hotel/fr/gite-la-mouet...,48.615795851658,-1.488698244809,Hébergement géré par un particulier,0,"9,3"
372,0,Mont Saint Michel,Vent des Grèves,https://www.booking.com/hotel/fr/vent-des-grev...,48.615403,-1.49144,"Offrant une vue sur le jardin, l’établissement...",0,"9,1"


In [11]:
df_weather = df_weather.rename(columns={
    "index": "city_id",
    "name": "city_name",
})
df_weather[df_weather["city_id"]== 0]


,city_id,city_name,temperature_mean,rain_mean,wind_score,score
0,0,Mont-Saint-Michel,7.56,0.83,0.98,5.75


In [12]:
df_global = pd.merge(df_weather, df_hotel, on='city_id', how='inner')
df_global = df_global.drop(columns=["city"])
df_global = df_global.rename(columns={"city_name": "city"})
df_global["stars"] = df_global["stars"].replace(0, np.nan)
df_global["rating"] = df_global["rating"].astype(str).str.replace(",", ".")
df_global["rating"] = pd.to_numeric(df_global["rating"], errors="coerce")
df_global.reset_index(drop=False, inplace=True)
df_global.rename(columns={"index": "Id"}, inplace=True)

df_global.sample(5)


,Id,city_id,city,temperature_mean,rain_mean,wind_score,score,hotel_name,hotel_link,lat,lon,desc,stars,rating
785,785,5,Paris,11.50,4.64,0.98,5.88,Le Mathurin Hotel & Spa,https://www.booking.com/hotel/fr/hotel-le-math...,48.87326461529425,2.3240315169095993,Le Mathurin Hotel & Spa est un hôtel 4 étoiles...,4.0,8.6
798,798,5,Paris,11.50,4.64,0.98,5.88,Select Hotel,https://www.booking.com/hotel/fr/select-paris....,48.84846716247429,2.3425568640232086,Le Select Hotel vous accueille au cœur de l'an...,4.0,9.0
599,599,32,Biarritz,15.27,3.53,1.73,10.01,Hôtel Barnea,https://www.booking.com/hotel/fr/hotelargieder...,43.48072811046226,-1.5642817318439484,L'Hotel Hôtel Barnea se situe dans le centre d...,3.0,9.0
445,445,11,Eguisheim,15.91,2.62,0.98,12.31,Résidence Pierre & Vacances Le Clos d'Eguisheim,https://www.booking.com/hotel/fr/residence-pie...,48.04502733567647,7.311854660511017,La Résidence Pierre & Vacances Le Clos d’Eguis...,3.0,8.0
125,125,21,Aix-en-Provence,19.14,0.14,1.43,17.57,Appartement Belle Lettre 1,https://www.booking.com/hotel/fr/appartement-b...,43.5282741,5.4523165,Hébergement géré par un particulier,NaN,8.6


In [13]:
df_global.to_csv("export/weather_and_hotels.csv", index=False)

## 4. Send data to S3
**Goal:** \
Enregistrer les données dans un bocket pour les conserver en sécurité.

**Actions:**
* Création d'un bucket sur S3
* Envoi et stockage des données dans ce bucket


In [14]:
session = boto3.Session()

In [15]:
s3 = session.resource("s3")

In [16]:
s3.Bucket("qha-kayak").upload_file("export/weather_and_hotels.csv", "weather_and_hotels.csv")

## 5. Send to database
**Goal:** \
Être en capacité de communiquer avec ma database.

**Actions:**
* Création d'une database en postgres
* Gestion des restrictions RDS et EC2
* Stockage des données d'environnement dans un fichier caché en local
* Envois des données à ma table


In [17]:
USERNAME = os.getenv("aws_kayak_user_name")
PASSWORD = os.getenv("aws_kayak_password")
HOSTNAME = os.getenv("aws_kayak_host_name")
DB_NAME = os.getenv("aws_kayak_db_name")
PORT = 5432

engine = create_engine(f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOSTNAME}:{PORT}/{DB_NAME}")

df_global.to_sql("weather_and_hotels", engine, if_exists="replace", index=False)


875

## 6. Importer les données de ma database et les analyser
**Goal:** \
Faire une analyse grâce au donnée récupérée.

**Actions:**
* Importation des données
* Structurer les données
* Obtenir le nom des villes avec une meilleure météo pour les 7 prochains jours
* Les afficher sur une carte
* Obtenir les données sur les 20 meilleurs hôtels dans ces villes
* Les afficher sur une carte


In [19]:
df_db = pd.read_sql("SELECT * FROM weather_and_hotels", engine)
df_db

,Id,city_id,city,temperature_mean,rain_mean,wind_score,score,hotel_name,hotel_link,lat,lon,desc,stars,rating
0,0,22,Avignon,20.17,0.16,1.29,18.72,Avignon Grand Hotel,https://www.booking.com/hotel/fr/avignongrandh...,43.94254505532995,4.804362058639526,"Doté d'une petite piscine sur le toit, l'Avign...",4.0,8.0
1,1,22,Avignon,20.17,0.16,1.29,18.72,Apartment 40m2-Air conditioning-Elevator-Under...,https://www.booking.com/hotel/fr/appart-parkin...,43.9451879,4.7996263,L’hébergement Apartment 40m2-Air conditioning-...,NaN,9.8
2,2,22,Avignon,20.17,0.16,1.29,18.72,Appart LeNA - Proche Théatres et Place des cor...,https://www.booking.com/hotel/fr/appart-lena-3...,43.9448251,4.8078226,"Doté d’une connexion Wi-Fi gratuite, l’héberge...",NaN,9.5
3,3,22,Avignon,20.17,0.16,1.29,18.72,Hôtel du Palais des Papes,https://www.booking.com/hotel/fr/du-palais-des...,43.95034694330067,4.8062074184417725,L'Hôtel du Palais des Papes est situé à Avigno...,3.0,7.2
4,4,22,Avignon,20.17,0.16,1.29,18.72,Maison d'hôtes L'îlot bambou,https://www.booking.com/hotel/fr/l-39-ilot-bam...,43.9510898394487,4.79801933009412,Hébergement géré par un particulier,NaN,9.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
870,870,2,Bayeux,10.83,5.82,0.98,4.03,Domaine de Bayeux,https://www.booking.com/hotel/fr/domaine-de-ba...,49.27232559999999,-0.6985101000000213,Le Domaine de Bayeux occupe une maison du XVII...,3.0,9.2
871,871,2,Bayeux,10.83,5.82,0.98,4.03,Premiere Classe Bayeux,https://www.booking.com/hotel/fr/premiere-clas...,49.269428715614055,-0.7066869735717773,Cet hôtel Première Classe se trouve à seulemen...,2.0,7.5
872,872,2,Bayeux,10.83,5.82,0.98,4.03,Gites les Pourquoi Pas - Résidence de Tourisme...,https://www.booking.com/hotel/fr/gites-les-pou...,49.2815226,-0.7081344,Proposant une connexion Wi-Fi gratuite et des ...,NaN,9.1
873,873,2,Bayeux,10.83,5.82,0.98,4.03,B&B Nathalie,https://www.booking.com/hotel/fr/imagine-bayeu...,49.276465,-0.706367,Situé à Bayeux et offrant une vue sur la ville...,NaN,9.5


In [22]:
df_weather['city_id'].head(5)

22    22
24    24
23    23
20    20
27    27
Name: city_id, dtype: int64

In [109]:
df_db["lat"] = pd.to_numeric(df_db["lat"], errors="coerce")
df_db["lon"] = pd.to_numeric(df_db["lon"], errors="coerce")
df_top5_cities_ids = (
    df_db
    .groupby(["city_id", "city"], as_index=False)
    .agg({
        "score": "mean",
        "lat": "mean",
        "lon": "mean",
        "temperature_mean": "mean",
        "rain_mean": "mean",
    })
    .sort_values("score", ascending=False)
    .head(5)
)
df_top5_cities_ids = df_top5_cities_ids.reset_index()
df_top5_cities_ids = df_top5_cities_ids.rename(columns={
    "score": "Conditions météo",
    "temperature_mean": "Température (°c)",
    "rain_mean": "Précipitations (mm)"
})
df_top5_cities_ids

,index,city_id,city,Conditions météo,lat,lon,Température (°c),Précipitations (mm)
0,22,22,Avignon,18.72,43.945685,4.808235,20.17,0.16
1,24,24,Nîmes,18.70,43.828190,4.361012,19.84,0.00
2,23,23,Uzès,17.77,44.011609,4.412413,19.24,0.19
3,20,20,Marseille,17.72,43.293980,5.375534,20.14,0.39
4,27,27,Collioure,17.70,42.524747,3.084555,19.05,0.37


In [110]:
fig = px.scatter_mapbox(df_top5_cities_ids,
                        lat="lat", lon="lon",
                        size="Conditions météo",
                        hover_name="city",
                        hover_data={
                            "Température (°c)": True,
                            "Précipitations (mm)": True,
                            "Conditions météo": False,
                            "lat": False,
                            "lon": False
                        },
                        color="Conditions météo",
                        color_continuous_scale="Plasma",
                        zoom=5,
                        center={"lat": 46.5, "lon": 2.5},
                        opacity=1.0,
                        mapbox_style="carto-positron",
                        title="Top 5 destinations météo en france pour les 7 prochains jours",
                        height=800,)

fig.update_layout(coloraxis_colorbar=dict(
    tickvals=[],
))

fig.show()

In [112]:
df_db_top5_cities_hotels = df_db[df_db['city_id'].isin(df_top5_cities_ids['city_id'])].copy()
df_db_top5_cities_hotels


,Id,city_id,city,temperature_mean,rain_mean,wind_score,score,hotel_name,hotel_link,lat,lon,desc,stars,rating
0,0,22,Avignon,20.17,0.16,1.29,18.72,Avignon Grand Hotel,https://www.booking.com/hotel/fr/avignongrandh...,43.942545,4.804362,"Doté d'une petite piscine sur le toit, l'Avign...",4.0,8.0
1,1,22,Avignon,20.17,0.16,1.29,18.72,Apartment 40m2-Air conditioning-Elevator-Under...,https://www.booking.com/hotel/fr/appart-parkin...,43.945188,4.799626,L’hébergement Apartment 40m2-Air conditioning-...,NaN,9.8
2,2,22,Avignon,20.17,0.16,1.29,18.72,Appart LeNA - Proche Théatres et Place des cor...,https://www.booking.com/hotel/fr/appart-lena-3...,43.944825,4.807823,"Doté d’une connexion Wi-Fi gratuite, l’héberge...",NaN,9.5
3,3,22,Avignon,20.17,0.16,1.29,18.72,Hôtel du Palais des Papes,https://www.booking.com/hotel/fr/du-palais-des...,43.950347,4.806207,L'Hôtel du Palais des Papes est situé à Avigno...,3.0,7.2
4,4,22,Avignon,20.17,0.16,1.29,18.72,Maison d'hôtes L'îlot bambou,https://www.booking.com/hotel/fr/l-39-ilot-bam...,43.951090,4.798019,Hébergement géré par un particulier,NaN,9.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,120,27,Collioure,19.05,0.37,0.98,17.70,Les Jasmins,https://www.booking.com/hotel/fr/chambre-d-39-...,42.523548,3.081874,La Chambre d'hôtes Les Jasmins est située à Co...,NaN,8.5
121,121,27,Collioure,19.05,0.37,0.98,17.70,La Chambre De Salome,https://www.booking.com/hotel/fr/la-chambre-de...,42.521117,3.077309,Hébergement géré par un particulier,NaN,8.6
122,122,27,Collioure,19.05,0.37,0.98,17.70,Studio Centre Plage parking gratuit,https://www.booking.com/hotel/fr/studio-centre...,42.522988,3.087330,L’hébergement Studio Centre Plage parking grat...,NaN,9.2
123,123,27,Collioure,19.05,0.37,0.98,17.70,Canta la Mar - Vue exceptionnelle,https://www.booking.com/hotel/fr/studio-canta-...,42.524431,3.090316,Hébergement géré par un particulier,NaN,9.5


In [127]:
df_db_top5_cities_hotels = df_db_top5_cities_hotels.sort_values(by='rating', ascending=False)
df_db_top20_hotel_in_top5_cities = df_db_top5_cities_hotels.head(20)
df_db_top20_hotel_in_top5_cities

,Id,city_id,city,temperature_mean,rain_mean,wind_score,score,hotel_name,hotel_link,lat,lon,desc,stars,rating
33,32,24,Nîmes,19.84,0.00,1.14,18.70,"AfroBoHome, Expérience Atypique, familles et p...",https://www.booking.com/hotel/fr/afrobohome-te...,43.827796,4.357863,"L’établissement AfroBoHome, Expérience Atypiqu...",NaN,10.0
1,1,22,Avignon,20.17,0.16,1.29,18.72,Apartment 40m2-Air conditioning-Elevator-Under...,https://www.booking.com/hotel/fr/appart-parkin...,43.945188,4.799626,L’hébergement Apartment 40m2-Air conditioning-...,NaN,9.8
110,110,27,Collioure,19.05,0.37,0.98,17.70,6BAT3 Appartement vue mer,https://www.booking.com/hotel/fr/residence-des...,42.524493,3.097216,"Bénéficiant d’un emplacement en bord de mer, l...",NaN,9.8
56,55,23,Uzès,19.24,0.19,1.28,17.77,La DAME de FLAUX,https://www.booking.com/hotel/fr/la-dame-de-fl...,44.012050,4.419244,Hébergement géré par un particulier,3.0,9.8
7,7,22,Avignon,20.17,0.16,1.29,18.72,Charme au cœur d'avignon,https://www.booking.com/hotel/fr/charme-au-coe...,43.947744,4.807970,Hébergement géré par un particulier,NaN,9.7
68,67,23,Uzès,19.24,0.19,1.28,17.77,La Grande Bourgade-Authentique Maison en Pierr...,https://www.booking.com/hotel/fr/la-grande-bou...,44.010459,4.417140,L’hébergement La Grande Bourgade-Authentique M...,NaN,9.7
118,118,27,Collioure,19.05,0.37,0.98,17.70,location Coma Chéric et parking,https://www.booking.com/hotel/fr/location-coma...,42.522981,3.087304,Hébergement géré par un particulier,NaN,9.6
31,30,24,Nîmes,19.84,0.00,1.14,18.70,"Maison ""COSY"" avec Parking Privé et Climatisation",https://www.booking.com/hotel/fr/le-cocon-nime...,43.833558,4.398547,Hébergement géré par un particulier,NaN,9.6
64,63,23,Uzès,19.24,0.19,1.28,17.77,Le Cerisier au cœur d'Uzès,https://www.booking.com/hotel/fr/le-cerisier-u...,44.010318,4.416918,Hébergement géré par un particulier,NaN,9.6
62,61,23,Uzès,19.24,0.19,1.28,17.77,La bohème place aux herbes,https://www.booking.com/hotel/fr/la-boheme-uze...,44.011676,4.419088,Hébergement géré par un particulier,NaN,9.5


In [126]:
fig = px.scatter_mapbox(df_db_top20_hotel_in_top5_cities,
                        lat="lat", lon="lon",
                        size="rating",
                        hover_name="hotel_name",
                        hover_data={
                            "rating": True,
                            "city": True,
                            "lat": False,
                            "lon": False
                        },
                        color="rating",
                        color_continuous_scale="plasma",
                        mapbox_style="carto-positron",
                        title="Top 20 hôtels dans les 5 meilleures villes",
                        height=800,)


fig.show()

In [129]:
df_top5_cities_names = df_top5_cities_ids['city'].tolist()
city_names_str = " ,".join(df_top5_cities_names)
print("Les villes ayant les meilleures conditions météo cette semaine sont :", city_names_str)

Les villes ayant les meilleures conditions météo cette semaine sont : Avignon ,Nîmes ,Uzès ,Marseille ,Collioure


## Résultats et perspectives

Grâce à ce projet, nous avons identifié les **villes françaises les plus favorables à un séjour cette semaine**, selon des critères de météo synthétisés en un **score de confort**.

Parmi les villes qui ressortent toutes combinent **faible pluie**, **vent modéré** et **températures agréables**.

Dans ces zones, Booking affiche une offre **riche en hôtels bien notés**, avec une forte concentration de 3 et 4 étoiles.  
La **corrélation entre météo favorable et offre hôtelière qualitative** permet d’envisager des campagnes ciblées ou une surcouche “suggestion intelligente” dans l’expérience utilisateur.

### Et ensuite ?

Ce projet pourrait être poussé plus loin en ajoutant :
- Des **critères tarifaires** (prix moyen, promotions)
- Des **filtres dynamiques** (via Streamlit par exemple)
- Une **API interne** qui renvoie les top destinations à afficher côté front

Cela permettrait à une plateforme comme Kayak ou Booking d’**augmenter le taux de clics sur des séjours pertinents** en fonction de la météo réelle.
